<div align="center">

# 🚕 Yandex Go Analytics

## 6 · Data Enrichment

*Join every trip with its driver's profile in a single, analysis-ready table.*

![layer](https://img.shields.io/badge/layer-gold-FFB300)
![operation](https://img.shields.io/badge/operation-LEFT%20JOIN-00ADD4)
![table](https://img.shields.io/badge/table-enriched_trips-2E7D32)

</div>

| ◀ Previous | 🏠 Home | Next ▶ |
|:--|:--:|--:|
| [5 · Aggregations & Optimizations](./5.%20Aggregations%20%26%20Optimizations.ipynb) | [Project Overview](./0.%20Project%20Overview.ipynb) | [7 · Refactor via Delta Live Tables](./7.%20Refactor%20via%20Delta%20Live%20Tables.ipynb) |

**Progress** &nbsp; ▰▰▰▰▰▰▱▱  **6 / 8**

---

## 🎯 Task requirements

- [x] Generate a **synthetic `driver_id`** for the taxi trips
- [x] Perform a **`LEFT JOIN`** between the trips and the driver profiles
- [x] Save the unified, enriched dataset as **`gold.enriched_trips`**

## 🧩 How it is solved

**Synthetic key.** The trip generator assigns every trip a `driver_id` drawn from a *deterministic* UUID pool
(fixed Faker seed). The driver generator uses the **same** pool, so trips and driver profiles share a join key
without any real-world identifiers.

```text
 silver.taxi  (stream, de-duplicated by trip id)
      │  driver_id
      ▼
   LEFT JOIN  ◀──────────  silver.drivers  (static snapshot, one row per driver id)
      │
      ▼
 gold.enriched_trips   =   all trip columns  +  driver_name · driver_car_number
                                                driver_experience · driver_rating [· driver_phone]
```

| Design choice | Why |
|:--|:--|
| `LEFT JOIN` | A trip must survive even when its driver profile is missing or was rejected by Silver |
| Stream ⨝ static | New trips flow in incrementally; the (small) driver dimension is read as a snapshot |
| `driver_` prefix | Keeps trip and driver columns unambiguous |
| `driver_phone` only if present | The `phone` column was rolled back in notebook 3, so it is added only if it exists in Silver |

## 1️⃣ Connect to Spark

In [6]:
from pyspark.sql import functions as F

from src.core.config import get_catalog, get_environment
from src.core.infra import DatabricksConnectionService

conn = DatabricksConnectionService(app_name="yandex-go-enrichment")
conn.connect()
spark = conn.spark

CATALOG = get_catalog()
print(f"✅ Connected | Spark {spark.version} | catalog = {CATALOG} | env = {get_environment()}")

2026-10-09 17:49:31.479 | INFO     | src.core.infra.connection.databricks_connector:connect:56 - SparkSession ready | app=yandex-go-enrichment | version=4.2.0 | runtime=serverless-connect


✅ Connected | Spark 4.2.0 | catalog = yandex_go_prod | env = prod


## 2️⃣ Build `gold.enriched_trips`

In [7]:
from src.core.etl.gold import EnrichedTripsGoldService

EnrichedTripsGoldService().run(spark)

enriched = spark.table(f"{CATALOG}.gold.enriched_trips")
print(f"gold.enriched_trips: {enriched.count():,} rows | {len(enriched.columns)} columns")
display(enriched.toPandas())

2026-10-09 17:49:31.493 | INFO     | src.core.etl.gold.trip_driver:run:34 - Reading stream from silver.taxi via checkpoints...
2026-10-09 17:50:17.253 | INFO     | src.core.etl.gold.trip_driver:run:73 - Enriched yandex_go_trips_pipeline successfully written!


gold.enriched_trips: 9,470 rows | 33 columns


,driver_id,id,user_id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,...,_ingested_at,_source_file,payment_type_description,_processed_at,driver_name,driver_car_number,driver_experience,driver_rating,_calculated_at,driver_phone
0,d5b9fe5c-f69b-4cf3-a8e1-a44aa0569d76,0c7ffaa1-c718-4800-a066-0e57aeb3f558,965befdf-6b4b-4411-8a4a-0e2df22b5b98,1,2026-10-09 06:54:16,2026-10-09 07:14:16,1,26.43,4,Y,...,2026-10-09 14:44:30.424,/Volumes/yandex_go_prod/raw_files/landing/parq...,Cash,2026-10-09 14:47:21.929,Richard Barnes,P165YB,8.0,4.62,2026-10-09 14:49:35.442,None
1,56b60bcd-9722-49a6-b094-29924a33f4fb,99aa9edb-2f1d-4b8f-89d7-5253636b3fc7,da743152-627b-41a1-bfd6-f23232ffe294,2,2026-10-06 06:06:00,2026-10-06 06:34:00,5,24.61,4,N,...,2026-10-09 14:44:30.424,/Volumes/yandex_go_prod/raw_files/landing/parq...,Credit Card,2026-10-09 14:47:21.929,Marcia Martinez,M609AK,5.0,4.50,2026-10-09 14:49:35.442,None
2,284fe16c-3fb0-4e60-af1d-54db32b81f27,ddad3942-0ef9-47f4-8684-c01d7776a171,4223623b-cc3e-4dde-9ad5-cf06364d7c87,2,2026-10-07 02:25:02,2026-10-07 02:44:02,4,0.76,5,Y,...,2026-10-09 14:44:30.424,/Volumes/yandex_go_prod/raw_files/landing/parq...,Credit Card,2026-10-09 14:47:21.929,Thomas Henry,M126HP,13.0,4.40,2026-10-09 14:49:35.442,(823)627-0324x9728
3,56b60bcd-9722-49a6-b094-29924a33f4fb,d3bbbcac-0b67-4e6b-a162-b8613efc865d,e87466d7-ad66-41bd-9367-6a024fdc6e1b,1,2026-10-09 09:54:12,2026-10-09 10:11:12,1,26.14,1,Y,...,2026-10-09 14:44:30.424,/Volumes/yandex_go_prod/raw_files/landing/parq...,Credit Card,2026-10-09 14:47:21.929,Marcia Martinez,M609AK,5.0,4.50,2026-10-09 14:49:35.442,None
4,fd35bbb7-fb13-4e8b-84dc-57de990e75ea,34a9cbe2-74b9-4ba7-852a-58179a24d139,e0ccedc5-f05d-476e-9a84-a51aa9d3d7c7,1,2026-10-01 07:31:37,2026-10-01 07:34:37,6,25.74,5,Y,...,2026-10-09 14:44:30.424,/Volumes/yandex_go_prod/raw_files/landing/parq...,Credit Card,2026-10-09 14:47:21.929,Meredith Richardson,A827AA,18.0,4.38,2026-10-09 14:49:35.442,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9465,fcc1b543-c49c-465b-96c2-703c4f353a13,d8e07133-f95b-47e7-9143-e7045b16cbc6,3ed8c56c-da09-4fa0-9282-8d8044b591f7,2,2026-10-04 14:08:35,2026-10-04 14:48:35,2,15.19,2,N,...,2026-10-09 14:35:45.565,/Volumes/yandex_go_prod/raw_files/landing/parq...,Other,2026-10-09 14:37:32.179,Patricia Long,O424XX,3.0,4.78,2026-10-09 14:49:35.442,None
9466,da3b3021-9e1d-4594-b168-ecb505d8ab8c,45aef693-48a3-423a-a6e7-27859ce8f6f1,de9e3757-5260-401e-aecf-67d2749176f4,2,2026-10-09 08:28:58,2026-10-09 08:53:58,6,13.57,1,N,...,2026-10-09 14:35:45.565,/Volumes/yandex_go_prod/raw_files/landing/parq...,Other,2026-10-09 14:37:32.179,Barbara Newton,E308MK,2.0,4.47,2026-10-09 14:49:35.442,None
9467,3a474975-30c3-4d97-984e-0f23b5eea8ed,add47caf-4aca-4a04-990f-0b8c11acde85,a6f2f7b8-0cf3-4b58-9910-8be58ce21ea3,2,2026-10-03 04:35:13,2026-10-03 04:53:13,2,11.24,2,Y,...,2026-10-09 14:35:45.565,/Volumes/yandex_go_prod/raw_files/landing/parq...,Credit Card,2026-10-09 14:37:32.179,None,None,NaN,NaN,2026-10-09 14:49:35.442,None
9468,81c1e7ff-6efa-4d5b-9988-5afcbb61a9cd,64ccb6b7-2a46-473b-b468-f1a17678e0dc,1327f1bc-2784-478f-b84f-16b3a79fbfaf,2,2026-10-02 08:23:26,2026-10-02 08:58:26,5,4.93,5,Y,...,2026-10-09 14:35:45.565,/Volumes/yandex_go_prod/raw_files/landing/parq...,Credit Card,2026-10-09 14:37:32.179,Benjamin Murphy,H415BA,24.0,4.30,2026-10-09 14:49:35.442,None


## 3️⃣ Verify the join

In [8]:
def check(label: str, condition: bool) -> None:
    print(("✅" if condition else "❌"), label)
    assert condition, label

silver_trip_count = spark.table(f"{CATALOG}.silver.taxi").select("id").distinct().count()
enriched_count = enriched.count()

check(
    f"row-preserving: {enriched_count:,} enriched rows == {silver_trip_count:,} distinct Silver trips",
    enriched_count == silver_trip_count,
)
check("one row per trip id", enriched_count == enriched.select("id").distinct().count())
check(
    "driver columns are present",
    {"driver_name", "driver_car_number", "driver_experience", "driver_rating"} <= set(enriched.columns),
)

✅ row-preserving: 9,470 enriched rows == 9,470 distinct Silver trips
✅ one row per trip id
✅ driver columns are present


### 🎯 Match rate

Trips whose driver has no Silver profile keep `NULL` driver columns — the signature of a `LEFT JOIN`.

In [9]:
match = (
    enriched.agg(
        F.count("*").alias("total_trips"),
        F.sum(F.when(F.col("driver_name").isNotNull(), 1).otherwise(0)).alias("with_driver"),
        F.sum(F.when(F.col("driver_name").isNull(), 1).otherwise(0)).alias("without_driver"),
    )
    .withColumn("match_rate_pct", F.round(F.col("with_driver") / F.col("total_trips") * 100, 1))
)
display(match.toPandas())

assert match.first()["with_driver"] > 0, "No trip was matched with a driver profile"
print("✅ Trips are enriched with driver profiles")

,total_trips,with_driver,without_driver,match_rate_pct
0,9470,7562,1908,79.9


✅ Trips are enriched with driver profiles


### 🏆 What the enriched table unlocks

Revenue per driver — impossible without the join.

In [10]:
top_drivers = (
    enriched.filter(F.col("driver_name").isNotNull())
    .groupBy("driver_id", "driver_name", "driver_experience", "driver_rating")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.sum("fare_amount"), 2).alias("revenue"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
    )
    .orderBy(F.col("revenue").desc())
)
display(top_drivers.toPandas())

,driver_id,driver_name,driver_experience,driver_rating,trips,revenue,avg_distance
0,9fb61a63-2069-4ab0-a33d-babce6fff3d9,William Walker,21,4.68,117,6374.94,16.22
1,cad5e297-e6b1-4665-8699-70498e2eac2a,Christian Schwartz,12,4.56,109,5910.52,15.71
2,90e25b2d-051d-4c86-aa8e-0a7c71085730,Amanda Henderson,21,4.85,105,5908.27,16.54
3,4b6587d8-8ef4-44d1-9299-b5fdfdff95ac,Cindy Li,20,4.98,107,5872.06,15.63
4,6576923c-81e1-4733-91dd-85fdb4164b76,Garrett Rodriguez,12,4.89,110,5840.52,15.24
...,...,...,...,...,...,...,...
86,fedc95a7-0c0e-496b-8150-f4d3df74d068,Nathaniel Hicks,15,4.64,55,2715.03,14.50
87,284fe16c-3fb0-4e60-af1d-54db32b81f27,Thomas Henry,13,4.40,56,2687.89,14.26
88,53a66ed1-dc80-4916-95d9-a429ad0cbe1d,Mike Holt,11,4.55,52,2642.65,15.02
89,81c1e7ff-6efa-4d5b-9988-5afcbb61a9cd,Benjamin Murphy,24,4.30,53,2558.97,14.32


---

## ✅ Checkpoint

- [x] Synthetic `driver_id` shared by trips and driver profiles
- [x] `LEFT JOIN` of `silver.taxi` with `silver.drivers`
- [x] `gold.enriched_trips` written; no trip lost, match rate reported

> **Next up ▶ [7 · Refactor via Delta Live Tables](./7.%20Refactor%20via%20Delta%20Live%20Tables.ipynb)** — recreate the whole pipeline declaratively with Delta Live Tables.

<div align="center"><sub>Yandex Go Analytics · notebook 6 of 8 · <a href="https://github.com/stanislavpanfilenko/YandexGoAnalytics">GitHub</a>